### Data Preprocessing


In [ ]:
# ensure dependencies are installed in the current Python environment
import sys
!{sys.executable} -m pip install -q --upgrade jupyterlab pandas tqdm pillow numpy torch torchvision facenet-pytorch imagehash opencv-python-headless faiss-cpu

In [23]:
# Config + imports
import os
from pathlib import Path
import hashlib
import json
from PIL import Image
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import imagehash
import cv2
import torch

# Paths: update these if needed
ROOT_DIR = Path("../data_raw")         # where your datasets live
OUT_DIR = Path("data_processed")    # where processed outputs will go
OUT_DIR.mkdir(parents=True, exist_ok=True)
(Path(OUT_DIR) / "aligned").mkdir(exist_ok=True)
(Path(OUT_DIR) / "embeddings").mkdir(exist_ok=True)
(Path(OUT_DIR) / "quarantine").mkdir(exist_ok=True)

# Run options
SAMPLE_LIMIT = None   # set to an int to process fewer images during testing
MIN_FACE_SIDE = 80    # min face side in pixels to accept (tweak for your data)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def to_rel(p: Path) -> str:
    """Return a stable relative path string from repo_root, fallback to name."""
    try:
        return str(p.relative_to(repo_root).as_posix())
    except Exception:
        try:
            return str(p.relative_to(Path.cwd()).as_posix())
        except Exception:
            return str(p.name)

Device: cpu


In [24]:
# Check that counts the number of files on disk (in this case, images)
# Disk counts per top-level folder
from pathlib import Path
exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pgm"}   # can be adjusted to count videos (.mp4, .avi, etc.)
ROOT = Path("../data_raw").resolve()   # same ROOT_DIR you used; adjust if needed
print("Scanning disk under:", to_rel(ROOT))
if not ROOT.exists():
    raise FileNotFoundError(f"{ROOT} not found from this kernel. Check ROOT_DIR.")

files = [p for p in ROOT.rglob("*") if p.is_file() and p.suffix.lower() in exts]
print("Total image files on disk:", len(files))

from collections import Counter
tops = [p.relative_to(ROOT).parts[0] if len(p.relative_to(ROOT).parts) else "" for p in files]
for k,v in Counter(tops).most_common():
    print(f"  {k}: {v}")

Scanning disk under: data_raw
Total image files on disk: 197693
  vggface2: 197693


In [26]:
# Manifest builder (per-dataset + master)
# - Scans ROOT_DIR for image files
# - Computes sha1, perceptual hash, image size, readable flag
# - Writes per-dataset manifest: data_processed/<dataset>/manifests/manifest_basic.csv
# - Writes master manifest: data_processed/manifests/master_manifest_basic.csv

import sys
from pathlib import Path
import hashlib
from PIL import Image, UnidentifiedImageError
import imagehash
import pandas as pd
from tqdm.auto import tqdm

# Config (adjust as needed)
ROOT_DIR = Path("../data_raw").resolve()        # where raw datasets live
OUT_ROOT = Path("../data_processed").resolve()  # output location for processed outputs
SAMPLE_LIMIT = None                             # set to an int for quick tests
exts = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".pgm", ".webp"}

repo_root = ROOT_DIR.parent.resolve()

OUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "manifests").mkdir(parents=True, exist_ok=True)

# used for detection of duplicates (i.e. bytewise identical files)
def sha1_of_file(path: Path, block_size: int = 65536):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(block_size), b""):
            h.update(b)
    return h.hexdigest()

# used for detection of near-duplicates (i.e. same image resized/recompressed)
def phash_of_image(path: Path):
    try:
        return str(imagehash.average_hash(Image.open(path)))
    except Exception:
        return None

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"ROOT_DIR not found: {ROOT_DIR}")

# Collect files (respect SAMPLE_LIMIT if set)
all_files = []
count = 0
for p in ROOT_DIR.rglob("*"):
    if not p.is_file():
        continue
    if p.suffix.lower() not in exts:
        continue
    all_files.append(p)
    count += 1
    if SAMPLE_LIMIT and count >= SAMPLE_LIMIT:
        break

print(f"Found {len(all_files)} image files under {to_rel(ROOT_DIR)}")

# Build master rows
master_rows = []

# Build per-dataset lists to write separate CSVs
per_dataset_rows = {}

for p in tqdm(sorted(all_files), desc="Scanning images"):
    try:
        rel = p.relative_to(ROOT_DIR)
    except Exception:
        # unexpected path; fallback to name only
        rel = Path(p.name)
    parts = rel.parts
    dataset = parts[0] if len(parts) > 0 else ""
    split = parts[1] if len(parts) > 1 else ""
    person_raw = parts[2] if len(parts) > 2 else p.parent.name
    person_id = person_raw   

    # Ensure per-dataset output directories exist
    ds_out = OUT_ROOT / dataset
    (ds_out / "aligned").mkdir(parents=True, exist_ok=True)
    (ds_out / "embeddings").mkdir(parents=True, exist_ok=True)
    (ds_out / "manifests").mkdir(parents=True, exist_ok=True)

    # Read image metadata
    w = h = fmt = None
    readable = False
    try:
        with Image.open(p) as im:
            w, h = im.size
            fmt = im.format
            readable = True
    except (UnidentifiedImageError, OSError, ValueError):
        readable = False
    except Exception:
        readable = False

    sha1 = None
    phash = None
    try:
        sha1 = sha1_of_file(p)
    except Exception:
        sha1 = None
    try:
        phash = phash_of_image(p)
    except Exception:
        phash = None

    row = {
        "image_path": str(p),        # path to original image
        "dataset": dataset,             # source labels (e.g. vggface2,ibeta)
        "split": split,                 # split (e.g. train, val, test)
        "person_id": person_id,         # person id (e.g. n000002)
        "person_raw": person_raw,       # raw person id extracted from file, so person_id can be renamed/reconstructed
        "sha1": sha1,                   # sha1 hash of file (checksum that is used to detect duplicates)
        "phash": phash,                 # perceptual hash of image (for near-duplicate detection)
        "width": w,                     # image width in pixels
        "height": h,                    # image height in pixels
        "format": fmt,                  # image format (e.g. JPEG, PNG)
        "readable": bool(readable)      # whether image could be opened/read
    }

    master_rows.append(row)
    per_dataset_rows.setdefault(dataset, []).append(row)

# Write master manifest
master_df = pd.DataFrame(master_rows)
master_path = OUT_ROOT / "manifests" / "master_manifest_basic.csv"
master_df.to_csv(master_path, index=False)
print(f"Wrote master manifest: {to_rel(master_path)} rows: {len(master_df)}")

# Write per-dataset manifests
for dataset, rows in per_dataset_rows.items():
    ds_manifest_path = OUT_ROOT / dataset / "manifests" / "manifest_basic.csv"
    pd.DataFrame(rows).to_csv(ds_manifest_path, index=False)
    print(f"Wrote {dataset} manifest: {to_rel(ds_manifest_path)} rows: {len(rows)}")

print("Manifest building complete.")

Found 197693 image files under data_raw


Scanning images: 100%|██████████| 197693/197693 [41:22<00:00, 79.63it/s] 


Wrote master manifest: data_processed/manifests/master_manifest_basic.csv rows: 197693
Wrote vggface2 manifest: data_processed/vggface2/manifests/manifest_basic.csv rows: 197693
Manifest building complete.
